# Vežbe 9: Word embeddings pomoću biblioteke PyTorch

U ovom notebook-u fokus je na predstavljanju reči pomoću embedding vektora. Primer je preuzet iz zvaničnog PyTorch [tutorijala](https://docs.pytorch.org/tutorials/beginner/nlp/word_embeddings_tutorial.html) za word embeddings.



In [1]:
# Import potrebnih biblioteka

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

## 1. Postavljanje seed-a za reproduktivnost rezultata

In [2]:
# Postaviti seed za reproduktivnost rezultata

torch.manual_seed(42)


## 2. Jednostavan primer embedding sloja

In [3]:
# Definisati mali rečnik

word_to_ix = {
    "hello": 0,
    "world": 1
}

In [4]:
# Kreirati embedding sloj

embeds = nn.Embedding(2, 5)

In [5]:
# Kreirati tenzor koji sadrži indeks reči hello

lookup_tensor = torch.tensor(
    [word_to_ix["hello"]],
    dtype=torch.long
)


In [6]:
# Prikazati embedding vektor za reč hello

hello_embed = embeds(lookup_tensor)

print(hello_embed)

tensor([[ 0.3367,  0.1288,  0.2345,  0.2303, -1.1229]],
       grad_fn=<EmbeddingBackward0>)


## 3. Definisanje teksta za n-gram language modeling

In [7]:
# Definisati tekst koji će se koristiti za trening modela

test_sentence = '''When forty winters shall besiege thy brow,
And dig deep trenches in thy beauty's field,
Thy youth's proud livery so gazed on now,
Will be a totter'd weed of small worth held:
Then being asked, where all thy beauty lies,
Where all the treasure of thy lusty days;
To say, within thine own deep sunken eyes,
Were an all-eating shame, and thriftless praise.
How much more praise deserv'd thy beauty's use,
If thou couldst answer This fair child of mine
Shall sum my count, and make my old excuse,
Proving his beauty by succession thine!
This were to be new made when thou art old,
And see thy blood warm when thou feelst it cold.'''.split()


# Prikazati broj reči i prvih nekoliko reči iz teksta

print("Broj reči u tekstu:", len(test_sentence))
print(test_sentence[:15])


Broj reči u tekstu: 115
['When', 'forty', 'winters', 'shall', 'besiege', 'thy', 'brow,', 'And', 'dig', 'deep', 'trenches', 'in', 'thy', "beauty's", 'field,']


In [8]:
# Definisati veličinu konteksta (2) i dimenziju embedding vektora (10)

CONTEXT_SIZE = 2
EMBEDDING_DIM = 10

## 4. Kreiranje n-gram trening primera

In [9]:
# Kreirati listu n-gram trening primera
# Svaki primer treba da ima oblik: ([prethodne reči], ciljna reč)

ngrams = []

for i in range(CONTEXT_SIZE, len(test_sentence)):

    context = []

    for j in range(CONTEXT_SIZE):
        context.append(test_sentence[i - j - 1])

    target = test_sentence[i]

    ngrams.append((context, target))

# Prikazati prva tri trening primera

print(ngrams[:3])


[(['forty', 'When'], 'winters'), (['winters', 'forty'], 'shall'), (['shall', 'winters'], 'besiege')]


## 6. Kreiranje rečnika

In [10]:
# Kreirati skup jedinstvenih reči

vocab = set(test_sentence)
print(len(vocab))

# Kreirati mapiranje reč -> indeks
word_to_ix = {word: i for i, word in enumerate(vocab)}

96


## 7. Definisanje modela

In [11]:
# Definisati model za n-gram language modeling

class NGramLanguageModeler(nn.Module):

    def __init__(self, vocab_size, embedding_dim, context_size):
        super().__init__()

        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear1 = nn.Linear(context_size * embedding_dim, 128)
        self.linear2 = nn.Linear(128, vocab_size)

    def forward(self, inputs):

        embeds = self.embeddings(inputs).reshape((1, -1))

        out = F.relu(self.linear1(embeds))

        out = self.linear2(out)

        return out


## 8. Kreiranje instance modela

In [12]:
# Kreirati instancu prethodno definisanog modela

model = NGramLanguageModeler(
    len(vocab),
    EMBEDDING_DIM,
    CONTEXT_SIZE
)

# Prikazati arhitekturu modela

print(model)


NGramLanguageModeler(
  (embeddings): Embedding(96, 10)
  (linear1): Linear(in_features=20, out_features=128, bias=True)
  (linear2): Linear(in_features=128, out_features=96, bias=True)
)


## 9. Definisanje funkcije greške i optimizatora

In [13]:
# Definisati funkciju greške

criterion = nn.CrossEntropyLoss()

# Definisati SGD optimizacioni algoritam

optimizer = optim.SGD(
    model.parameters(),
    lr=0.001
)


## 10. Trening modela

In [14]:
# Istrenirati model i sačuvati vrednosti greške po epohama

num_epochs = 10

losses = []

for epoch in range(num_epochs):

    total_loss = 0

    for context, target in ngrams:

        context_idxs = torch.tensor(
            [word_to_ix[w] for w in context],
            dtype=torch.long
        )

        target_tensor = torch.tensor(
            [word_to_ix[target]],
            dtype=torch.long
        )


        outputs = model(context_idxs)

        loss = criterion(outputs, target_tensor)

        model.zero_grad()

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    losses.append(total_loss)
    print(f"Epoha {epoch+1}: greška = {total_loss:.4f}")



Epoha 1: greška = 518.9994
Epoha 2: greška = 516.9041
Epoha 3: greška = 514.8210
Epoha 4: greška = 512.7503
Epoha 5: greška = 510.6905
Epoha 6: greška = 508.6403
Epoha 7: greška = 506.5992
Epoha 8: greška = 504.5664
Epoha 9: greška = 502.5414
Epoha 10: greška = 500.5238


## 11. Prikaz vrednosti greške po epohama

In [15]:
# Prikazati tabelarno vrednosti greške po epohama

results = pd.DataFrame({
    "Epoha": range(1, num_epochs + 1),
    "Loss": losses
})

print(results)


   Epoha        Loss
0      1  518.999390
1      2  516.904068
2      3  514.821009
3      4  512.750317
4      5  510.690547
5      6  508.640341
6      7  506.599186
7      8  504.566371
8      9  502.541423
9     10  500.523781


## 12. Prikaz embedding vektora za jednu reč

In [16]:
# Prikazati embedding vektor za reč beauty

embedding_vector = model.embeddings.weight[word_to_ix["beauty"]]

print(embedding_vector)


tensor([ 2.8125,  0.3610, -0.0903,  0.4589,  0.5352,  0.5240,  1.1414,  0.0512,
         0.7286, -0.7110], grad_fn=<SelectBackward0>)


## 14. Priprema podataka za CBOW model

In [17]:
# Definisati veličinu konteksta
# Koriste se dve reči levo i dve reči desno od ciljne reči

CONTEXT_SIZE = 2

# Definisati tekst za CBOW primer

raw_text = '''We are about to study the idea of a computational process.
Computational processes are abstract beings that inhabit computers.
As they evolve processes manipulate other abstract things called data.
The evolution of a process is directed by a pattern of rules
called a program. People create programs to direct processes.'''.split()

# Kreirati rečnik

vocab = set(raw_text)
vocab_size = len(vocab)

word_to_ix = {word: i for i, word in enumerate(vocab)}

# Kreirati CBOW trening primere
# Svaki primer treba da ima oblik: ([reči iz okruženja], ciljna reč)

data = []

for i in range(CONTEXT_SIZE, len(raw_text) - CONTEXT_SIZE):

    context = []

    for j in range(CONTEXT_SIZE):
        context.append(raw_text[i - j - 1])

    for j in range(CONTEXT_SIZE):
        context.append(raw_text[i + j + 1])

    target = raw_text[i]

    data.append((context, target))

# Prikazati prvih pet primera

print(data[:5])


[(['are', 'We', 'to', 'study'], 'about'), (['about', 'are', 'study', 'the'], 'to'), (['to', 'about', 'the', 'idea'], 'study'), (['study', 'to', 'idea', 'of'], 'the'), (['the', 'study', 'of', 'a'], 'idea')]


## 15. Funkcija za pretvaranje konteksta u tensor

In [18]:
# Definisati funkciju koja reči iz konteksta pretvara u tensor indeksa

def make_context_vector(context, word_to_ix):

    idxs = [word_to_ix[w] for w in context]

    return torch.tensor(idxs, dtype=torch.long)


In [19]:
# Prikazati primer pretvaranja konteksta u tensor

context_vector = make_context_vector(data[0][0], word_to_ix)

print("Kontekst:", data[0][0])
print("Tensor indeksa:", context_vector)


Kontekst: ['are', 'We', 'to', 'study']
Tensor indeksa: tensor([23, 31, 14, 18])
